# NLP Sentiment Analysis — Financial News Headlines

**Goal:** Classify financial sentences as `positive`, `negative`, or `neutral` using classical NLP and ML baselines.

**Dataset:** Financial PhraseBank (Hugging Face) — ~4,800 labeled financial sentences.

---
**Notebook Sections:**
1. Imports & Setup
2. Data Loading & Exploration
3. Text Preprocessing
4. Feature Extraction (TF-IDF)
5. Model Training
6. Evaluation
7. Error Analysis

## 1. Imports & Setup

In [ ]:

%pip install nltk
%pip install datasets wordcloud

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import warnings
warnings.filterwarnings('ignore')

# NLP
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# ML
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# Dataset
from datasets import load_dataset

# Visualization
from wordcloud import WordCloud

# --- FIX: Bypass SSL Verification for Downloads ---
import ssl
try:
    _create_unverified_https_context = ssl._create_unverified_context
except AttributeError:
    pass
else:
    ssl._create_default_https_context = _create_unverified_https_context
# --------------------------------------------------

# Download required NLTK data
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

# Plot style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('Set2')

print('All imports and data downloads successful.')

## 2. Data Loading & Exploration

In [ ]:
# Load Financial PhraseBank from a clean, script-free Parquet repository
dataset = load_dataset('lmassaron/FinancialPhraseBank')

# Convert to DataFrame
df = pd.DataFrame(dataset['train'])

# In this clean version, the labels are already text strings ('positive', 'neutral', 'negative')
# Let's map them to a uniform column name to keep the rest of your notebook intact
df['sentiment'] = df['sentiment']

print(f'Dataset shape: {df.shape}')
df.head(10)

In [ ]:
# Class distribution
print('Class Distribution:')
print(df['sentiment'].value_counts())
print()
print(f'Class balance (%):')
print(df['sentiment'].value_counts(normalize=True).mul(100).round(1))

In [ ]:
# Plot class distribution
fig, ax = plt.subplots(figsize=(7, 4))
counts = df['sentiment'].value_counts()
bars = ax.bar(counts.index, counts.values, color=['#e74c3c', '#95a5a6', '#2ecc71'], edgecolor='white', linewidth=1.5)

for bar, val in zip(bars, counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 10, str(val),
            ha='center', va='bottom', fontweight='bold', fontsize=11)

ax.set_title('Class Distribution — Financial PhraseBank', fontsize=13, fontweight='bold')
ax.set_xlabel('Sentiment')
ax.set_ylabel('Count')
plt.tight_layout()
plt.savefig('../data/class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Sentence length analysis
df['sentence_length'] = df['sentence'].apply(lambda x: len(x.split()))

print('Sentence Length Stats (word count):')
print(df.groupby('sentiment')['sentence_length'].describe().round(1))

fig, ax = plt.subplots(figsize=(8, 4))
for label in ['negative', 'neutral', 'positive']:
    subset = df[df['sentiment'] == label]['sentence_length']
    ax.hist(subset, alpha=0.6, bins=30, label=label)

ax.set_title('Sentence Length Distribution by Sentiment', fontsize=13, fontweight='bold')
ax.set_xlabel('Word Count')
ax.set_ylabel('Frequency')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Sample sentences per class
print('=== SAMPLE SENTENCES ===')
for sentiment in ['positive', 'neutral', 'negative']:
    print(f'\n--- {sentiment.upper()} ---')
    samples = df[df['sentiment'] == sentiment]['sentence'].sample(3, random_state=42)
    for i, s in enumerate(samples, 1):
        print(f'{i}. {s}')

## 3. Text Preprocessing

In [ ]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    """
    Full preprocessing pipeline:
    1. Lowercase
    2. Remove punctuation and special characters
    3. Remove extra whitespace
    4. Remove stopwords
    5. Lemmatize
    """
    # Lowercase
    text = text.lower()
    
    # Remove punctuation and special characters (keep only letters and spaces)
    text = re.sub(r'[^a-z\s]', '', text)
    
    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    
    # Tokenize, remove stopwords, lemmatize
    tokens = text.split()
    tokens = [lemmatizer.lemmatize(t) for t in tokens if t not in stop_words and len(t) > 2]
    
    return ' '.join(tokens)

# Apply preprocessing
df['cleaned'] = df['sentence'].apply(preprocess_text)

# Before / After comparison
print('Preprocessing Examples:')
print('='*70)
for _, row in df.sample(3, random_state=1).iterrows():
    print(f'ORIGINAL : {row["sentence"]}')
    print(f'CLEANED  : {row["cleaned"]}')
    print(f'LABEL    : {row["sentiment"]}')
    print('-'*70)

In [ ]:
# Word clouds per sentiment class
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
colors = {'negative': 'Reds', 'neutral': 'Blues', 'positive': 'Greens'}

for ax, sentiment in zip(axes, ['negative', 'neutral', 'positive']):
    text = ' '.join(df[df['sentiment'] == sentiment]['cleaned'])
    wc = WordCloud(width=400, height=300, background_color='white',
                   colormap=colors[sentiment], max_words=60).generate(text)
    ax.imshow(wc, interpolation='bilinear')
    ax.set_title(f'{sentiment.upper()}', fontsize=12, fontweight='bold')
    ax.axis('off')

plt.suptitle('Most Frequent Words by Sentiment Class', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../data/wordclouds.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Feature Extraction (TF-IDF)

In [ ]:
# Train/test split
X = df['cleaned']
y = df['label']  # 0=negative, 1=neutral, 2=positive

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Train size: {len(X_train)}')
print(f'Test size : {len(X_test)}')
print(f'\nTrain class distribution:')
print(pd.Series(y_train).value_counts())

In [ ]:
# TF-IDF Vectorizer
# unigrams + bigrams, top 5000 features, sublinear TF scaling
tfidf = TfidfVectorizer(
    ngram_range=(1, 2),
    max_features=5000,
    sublinear_tf=True,
    min_df=2
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print(f'TF-IDF matrix shape (train): {X_train_tfidf.shape}')
print(f'TF-IDF matrix shape (test) : {X_test_tfidf.shape}')

In [ ]:
# Top TF-IDF features
feature_names = tfidf.get_feature_names_out()
print(f'Total features: {len(feature_names)}')
print(f'Sample features: {feature_names[:20]}')

## 5. Model Training

In [ ]:
# --- Model 1: Logistic Regression ---
lr = LogisticRegression(max_iter=1000, C=1.0, random_state=42)
lr.fit(X_train_tfidf, y_train)

y_pred_lr = lr.predict(X_test_tfidf)
print(f'Logistic Regression Accuracy: {accuracy_score(y_test, y_pred_lr):.4f}')

In [ ]:
# --- Model 2: Multinomial Naive Bayes ---
nb = MultinomialNB(alpha=0.1)
nb.fit(X_train_tfidf, y_train)

y_pred_nb = nb.predict(X_test_tfidf)
print(f'Naive Bayes Accuracy: {accuracy_score(y_test, y_pred_nb):.4f}')

## 6. Evaluation

In [ ]:
# Classification reports
target_names = ['negative', 'neutral', 'positive']

print('=== LOGISTIC REGRESSION ===')
print(classification_report(y_test, y_pred_lr, target_names=target_names))

print('=== MULTINOMIAL NAIVE BAYES ===')
print(classification_report(y_test, y_pred_nb, target_names=target_names))

In [ ]:
# Confusion matrices
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, y_pred, title in zip(axes,
                              [y_pred_lr, y_pred_nb],
                              ['Logistic Regression', 'Naive Bayes']):
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=target_names, yticklabels=target_names)
    ax.set_title(f'{title}\nAccuracy: {accuracy_score(y_test, y_pred):.3f}',
                 fontsize=12, fontweight='bold')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')

plt.suptitle('Confusion Matrices — Sentiment Classification', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../data/confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Model comparison bar chart
from sklearn.metrics import f1_score

models = ['Logistic Regression', 'Naive Bayes']
accuracies = [
    accuracy_score(y_test, y_pred_lr),
    accuracy_score(y_test, y_pred_nb)
]
f1_scores = [
    f1_score(y_test, y_pred_lr, average='weighted'),
    f1_score(y_test, y_pred_nb, average='weighted')
]

x = np.arange(len(models))
width = 0.35

fig, ax = plt.subplots(figsize=(7, 4))
bars1 = ax.bar(x - width/2, accuracies, width, label='Accuracy', color='#3498db')
bars2 = ax.bar(x + width/2, f1_scores, width, label='Weighted F1', color='#2ecc71')

for bar in bars1 + bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=10)

ax.set_ylim(0, 1.0)
ax.set_xticks(x)
ax.set_xticklabels(models)
ax.set_title('Model Comparison — Accuracy vs Weighted F1', fontsize=12, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig('../data/model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Error Analysis

In [ ]:
# Identify misclassified examples (using Logistic Regression)
# Convert X_test series to a base DataFrame
test_df = X_test.reset_index(drop=True).to_frame()


test_df['actual'] = y_test.reset_index(drop=True)
test_df['predicted'] = pd.Series(y_pred_lr)


test_df['original'] = X_test.reset_index(drop=True)


errors = test_df[test_df['actual'] != test_df['predicted']].copy()

print(f'Total misclassified: {len(errors)} / {len(test_df)} ({len(errors)/len(test_df)*100:.1f}%)')

In [ ]:
# Most common error patterns
print('Error Pattern Counts (Actual → Predicted):')
print(errors.groupby(['actual', 'predicted']).size().sort_values(ascending=False))

In [ ]:
print('=== SAMPLE MISCLASSIFICATIONS ===')
print_map = {0: 'negative', 1: 'neutral', 2: 'positive'}

for (actual, predicted), group in errors.groupby(['actual', 'predicted']):
    pred_str = print_map.get(predicted, predicted)
    print(f'\n--- Actual: {str(actual).upper()} | Predicted: {str(pred_str).upper()} ---')
    for _, row in group.head(2).iterrows():
        print(f'  "{row["original"]}"')

In [ ]:
# Top features per class (Logistic Regression coefficients)
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for i, (ax, sentiment) in enumerate(zip(axes, target_names)):
    coefs = lr.coef_[i]
    top_indices = np.argsort(coefs)[-15:][::-1]
    top_features = [feature_names[j] for j in top_indices]
    top_coefs = coefs[top_indices]
    
    colors = ['#2ecc71' if c > 0 else '#e74c3c' for c in top_coefs]
    ax.barh(range(len(top_features)), top_coefs, color=colors)
    ax.set_yticks(range(len(top_features)))
    ax.set_yticklabels(top_features, fontsize=9)
    ax.set_title(f'Top Features: {sentiment.upper()}', fontweight='bold')
    ax.invert_yaxis()

plt.suptitle('Most Influential Words per Sentiment Class (LR Coefficients)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('../data/feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

## Summary

| Model | Accuracy | Weighted F1 |
|---|---|---|
| Logistic Regression | — | — |
| Multinomial Naive Bayes | — | — |

**Key findings:**
- *Fill in after running*

**Observations from error analysis:**
- *Fill in after running*

**Next steps:**
- Experiment with `C` parameter in Logistic Regression
- Try SVM (`LinearSVC`) as a third baseline
- Explore FinBERT (pretrained transformer fine-tuned on financial text) for a significant accuracy boost